# Домашняя работа: создание нейро-сотрудника

## Тема проекта

Я создаю нейро-сотрудника под названием **«Нейро-помощник программиста»**.

Этот нейро-сотрудник помогает фриланс-разработчику анализировать заявки клиентов, составлять уточняющие вопросы, предлагать техническое решение, оценивать сложность, сроки, стоимость и готовить профессиональный ответ заказчику.

## Личные данные нейро-сотрудника

Имя: Артём  
Возраст: 32 года  
Должность: технический аналитик и помощник фриланс-разработчика  
Опыт: 8 лет в разработке, автоматизации, Telegram-ботах, API-интеграциях и MVP-проектах  

## Целевая аудитория

Целевая аудитория — начинающие и практикующие фрилансеры-разработчики, которые получают заявки от клиентов и хотят быстро понять, что нужно сделать, какие вопросы задать и как грамотно ответить заказчику.

## Какие задачи решает нейро-сотрудник

Артём помогает:
- определить тип проекта;
- понять, что хочет клиент;
- выделить уже понятные требования;
- найти неясные места;
- составить уточняющие вопросы;
- предложить техническое решение;
- оценить сложность проекта;
- предложить примерные сроки и стоимость;
- предупредить о рисках;
- написать готовый ответ клиенту.

## База знаний

Для работы нейро-сотрудника используется база знаний в виде Google Документа.

Сначала база знаний была создана в плохо структурированном виде: обычным текстом были описаны типы заказов, вопросы для уточнения, сроки, стоимость и риски.

Затем база знаний была структурирована по двухуровневому плану:

- заголовок 1 уровня - крупная тема;
- заголовок 2 уровня - конкретный раздел;
- фрагмент - полезная информация для ответа модели.

Такая структура нужна для улучшения качества поиска в RAG-системе. Когда пользователь задаёт вопрос, векторная база находит не весь документ целиком, а наиболее подходящие фрагменты.

## Комментарий о структуризации с помощью ChatGPT

Для структуризации базы знаний я использовал ChatGPT.

Я попросил ChatGPT преобразовать плохо структурированный текст в двухуровневую структуру:

1. Заголовок 1 уровня - логическая тема.
2. Заголовок 2 уровня - смысловая группа внутри темы.
3. Фрагмент - исходный или улучшенный текст из базы знаний.

Пример запроса к ChatGPT:

"Структурируй этот текст для базы знаний нейро-сотрудника. Сделай двухуровневый план. Каждый блок оформи в формате: заголовок 1 уровня, заголовок 2 уровня, фрагмент. Нужно, чтобы эти фрагменты потом можно было подавать в языковую модель через RAG."

После структуризации база знаний стала более удобной для поиска. Это должно повысить качество ответов нейро-сотрудника.

In [1]:
#Библиотеки
!pip -q install openai gradio tiktoken requests chromadb langchain langchain-openai langchain-chroma langchain-text-splitters

In [2]:
import os
import getpass

os.environ.pop("OPENAI_API_KEY", None)

api_key = getpass.getpass("Введите новый OpenAI Secret Key: ").strip()

if not api_key.startswith(("sk-", "sk-proj-")):
    raise ValueError("Вы вставили не Secret Key. Нужен ключ, который начинается с sk- или sk-proj-.")

os.environ["OPENAI_API_KEY"] = api_key

print("Ключ принят по формату. Можно продолжать.")

Введите новый OpenAI Secret Key: ··········
Ключ принят по формату. Можно продолжать.


In [3]:
#настройки нейро-сотрудника
DOC_URL = "https://docs.google.com/document/d/1u2b7fZs0ghR8-Ts6NoNfe1hJV2WzMLS7U6thIYJWAQ4/edit?usp=sharing"

models = [
    {
        "name": "Нейро-помощник программиста",
        "doc": DOC_URL,
        "prompt": """
Ты — нейро-сотрудник Артём, 32 года.

Ты работаешь техническим аналитиком и помощником фриланс-разработчика.
Твоя специализация — анализ заявок клиентов на разработку, автоматизацию, Telegram-ботов, API-интеграции, сайты, парсеры, CRM и MVP-проекты.

Твоя целевая аудитория — начинающие и практикующие фрилансеры-разработчики, которым нужно быстро понять задачу клиента и профессионально ответить на заявку.

Твои обязанности:
1. Анализировать текст заявки клиента.
2. Определять тип проекта.
3. Выделять, что именно хочет клиент.
4. Находить неясные места в задаче.
5. Формировать список уточняющих вопросов.
6. Предлагать техническое решение простым языком.
7. Оценивать сложность проекта: простая, средняя, сложная.
8. Давать примерную оценку сроков и стоимости на основе базы знаний.
9. Предупреждать о рисках.
10. Готовить готовый текст ответа клиенту.

Правила работы:
- Отвечай только на основе базы знаний и найденных фрагментов.
- Если информации недостаточно, честно напиши, что нужно уточнить.
- Не придумывай точные сроки и цены, если в базе знаний нет подходящего примера.
- Пиши профессионально, спокойно и уверенно.
- Не используй лишнюю воду.
- Не обещай клиенту невозможное.
- Всегда предупреждай о рисках, если они есть.

Формат ответа:
1. Тип проекта
2. Что хочет клиент
3. Что уже понятно
4. Что нужно уточнить
5. Предлагаемое техническое решение
6. Оценка сложности
7. Примерные сроки
8. Примерная стоимость
9. Возможные риски
10. Готовый ответ клиенту

Документ с информацией для ответа:
""",
        "query": """
Клиент написал:
Здравствуйте. Нужна интеграция Instantly.ai с Google Таблицей. Сейчас ответы клиентов приходят в личный кабинет Instantly, а мне нужно, чтобы когда клиент ответил на письмо, его ответ и данные автоматически записывались в Google Таблицу. Можно через webhook, Google Apps Script, Make или Zapier. Нужно понять сроки и стоимость.
"""
    }
]

In [4]:
#класс нейро-сотрудника
import os
import re
import requests
import tiktoken
from openai import OpenAI

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter


class NeuroEmployee:
    def __init__(self, model="gpt-4o-mini"):
        self.model = model
        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.search_index = None
        self.log = ""

    def extract_google_doc_id(self, url):
        match = re.search(r"/document/d/([a-zA-Z0-9-_]+)", url)
        if match is None:
            raise ValueError("Неверная ссылка на Google Docs. Проверьте, что это ссылка именно на документ Google Docs.")
        return match.group(1)

    def download_google_doc_text(self, url):
        doc_id = self.extract_google_doc_id(url)
        export_url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"

        response = requests.get(export_url)
        response.raise_for_status()

        text = response.text.strip()

        if len(text) < 100:
            raise ValueError(
                "Документ скачался пустым или слишком коротким. "
                "Проверьте доступ: Google Docs должен быть открыт для всех, у кого есть ссылка."
            )

        return text

    def parse_structured_knowledge(self, text):
        """
        Парсим структурированный документ формата:
        # Заголовок 1 уровня
        ## Заголовок 2 уровня
        Фрагмент: текст

        На выходе получаем список Document, где каждый фрагмент имеет:
        Заголовок 1 уровня
        Заголовок 2 уровня
        Фрагмент
        """

        documents = []

        current_h1 = "Без заголовка 1 уровня"
        current_h2 = "Без заголовка 2 уровня"
        buffer = []

        def flush_buffer():
            nonlocal buffer, current_h1, current_h2, documents

            fragment = "\n".join(buffer).strip()

            if fragment:
                page_content = f"""Заголовок 1 уровня: {current_h1}
Заголовок 2 уровня: {current_h2}
Фрагмент: {fragment}"""

                documents.append(
                    Document(
                        page_content=page_content,
                        metadata={
                            "h1": current_h1,
                            "h2": current_h2
                        }
                    )
                )

            buffer = []

        lines = text.splitlines()

        for line in lines:
            line = line.strip()

            if not line:
                continue

            if line.startswith("# ") and not line.startswith("## "):
                flush_buffer()
                current_h1 = line.replace("# ", "").strip()
                current_h2 = "Без заголовка 2 уровня"

            elif line.startswith("## "):
                flush_buffer()
                current_h2 = line.replace("## ", "").strip()

            else:
                buffer.append(line)

        flush_buffer()

        structured_docs = [
            doc for doc in documents
            if doc.metadata["h1"] != "Без заголовка 1 уровня"
        ]

        return structured_docs

    def fallback_split_text(self, text):
        """
        Запасной вариант, если структурированные фрагменты не найдены.
        Делим обычный текст на куски.
        """

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=150,
            separators=["\n\n", "\n", ". ", " "]
        )

        chunks = splitter.split_text(text)

        documents = []
        for i, chunk in enumerate(chunks):
            page_content = f"""Заголовок 1 уровня: Неструктурированная база знаний
Заголовок 2 уровня: Фрагмент {i + 1}
Фрагмент: {chunk}"""

            documents.append(
                Document(
                    page_content=page_content,
                    metadata={
                        "h1": "Неструктурированная база знаний",
                        "h2": f"Фрагмент {i + 1}"
                    }
                )
            )

        return documents

    def num_tokens_from_string(self, text):
        try:
            encoding = tiktoken.encoding_for_model(self.model)
        except Exception:
            encoding = tiktoken.get_encoding("cl100k_base")

        return len(encoding.encode(text))

    def load_search_indexes(self, url):
        self.log = ""

        self.log += "1. Скачиваем Google Документ...\n"
        text = self.download_google_doc_text(url)
        self.log += f"Документ скачан. Длина текста: {len(text)} символов.\n\n"

        self.log += "2. Пробуем разобрать документ как структурированную базу знаний...\n"
        documents = self.parse_structured_knowledge(text)

        if len(documents) == 0:
            self.log += "Структурированные фрагменты не найдены. Используем обычное разбиение на чанки.\n"
            documents = self.fallback_split_text(text)
        else:
            self.log += f"Найдено структурированных фрагментов: {len(documents)}.\n"

        all_text = "\n\n".join([doc.page_content for doc in documents])
        token_count = self.num_tokens_from_string(all_text)
        self.log += f"Количество токенов в базе знаний: {token_count}.\n\n"

        self.log += "3. Примеры фрагментов, которые попадут в векторную базу:\n\n"

        for i, doc in enumerate(documents[:5], start=1):
            self.log += f"--- Фрагмент {i} ---\n"
            self.log += doc.page_content[:1000] + "\n\n"

        self.log += "4. Создаём эмбеддинги и загружаем данные в ChromaDB...\n"

        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        self.search_index = Chroma.from_documents(
            documents=documents,
            embedding=embeddings
        )

        self.log += "Готово. База знаний загружена в векторную базу данных.\n"

        return self.log

    def answer(self, system_prompt, user_query, k=5, temperature=0.2):
        if self.search_index is None:
            return "", "Сначала нажмите кнопку «Обучить модель»."

        self.log += "\n\n5. Ищем релевантные фрагменты в векторной базе...\n"

        docs = self.search_index.similarity_search(user_query, k=k)

        context = "\n\n".join(
            [f"Найденный фрагмент №{i + 1}:\n{doc.page_content}" for i, doc in enumerate(docs)]
        )

        self.log += "\nФрагменты, переданные в языковую модель:\n\n"
        self.log += context + "\n\n"

        messages = [
            {
                "role": "system",
                "content": system_prompt + "\n\n" + context
            },
            {
                "role": "user",
                "content": user_query
            }
        ]

        self.log += f"Количество найденных фрагментов: {len(docs)}.\n"

        completion = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature
        )

        answer = completion.choices[0].message.content

        self.log += "\n6. Ответ языковой модели получен.\n"
        self.log += f"Использовано токенов на запрос: {completion.usage.prompt_tokens}\n"
        self.log += f"Использовано токенов всего: {completion.usage.total_tokens}\n"

        return answer, self.log

In [5]:
# Контрольная проверка работы нейро-сотрудника без Gradio

test_neuro = NeuroEmployee(model="gpt-4o-mini")

print("Обучаем нейро-сотрудника...")
train_log = test_neuro.load_search_indexes(DOC_URL)
print(train_log)

print("\nОтправляем тестовый запрос...")
test_answer, test_log = test_neuro.answer(
    models[0]["prompt"],
    models[0]["query"]
)

print("\nОтвет нейро-сотрудника:\n")
print(test_answer)

Обучаем нейро-сотрудника...
1. Скачиваем Google Документ...
Документ скачан. Длина текста: 8938 символов.

2. Пробуем разобрать документ как структурированную базу знаний...
Найдено структурированных фрагментов: 20.
Количество токенов в базе знаний: 1674.

3. Примеры фрагментов, которые попадут в векторную базу:

--- Фрагмент 1 ---
Заголовок 1 уровня: Анализ клиентских заявок
Заголовок 2 уровня: Общий принцип анализа заявки
Фрагмент: Фрагмент: Фриланс-разработчик часто получает заявки от клиентов в свободной форме. Клиенты редко пишут полноценное техническое задание. Обычно они описывают задачу коротко: нужен бот, нужна интеграция, нужен сайт, нужно связать сервисы, нужно сделать автоматизацию. Поэтому сначала нужно понять, какой тип проекта перед нами.

--- Фрагмент 2 ---
Заголовок 1 уровня: Типы проектов
Заголовок 2 уровня: Telegram-боты
Фрагмент: Фрагмент: Если клиент просит Telegram-бота, нужно уточнить роли пользователей, основные сценарии, нужна ли база данных, нужна ли админ-пан

In [ ]:
#Интерфейс Gradio
import gradio as gr

neuro = NeuroEmployee(model="gpt-4o-mini")


def onchange_model(dropdown_index):
    item = models[dropdown_index]

    return (
        item["name"],
        item["prompt"],
        item["query"],
        f"<a href='{item['doc']}' target='_blank'>Открыть Google Документ с базой знаний</a>"
    )


def train_model(dropdown_index):
    item = models[dropdown_index]
    return neuro.load_search_indexes(item["doc"])


def ask_model(prompt, query):
    answer, log = neuro.answer(prompt, query)
    return answer, log


with gr.Blocks() as demo:
    gr.Markdown("# Нейро-помощник программиста")
    gr.Markdown(
        """
        Этот нейро-сотрудник помогает фриланс-разработчику анализировать заявки клиентов,
        составлять уточняющие вопросы, оценивать сложность, сроки, стоимость и писать готовый ответ заказчику.
        """
    )

    subject = gr.Dropdown(
        choices=[(item["name"], i) for i, item in enumerate(models)],
        value=0,
        label="Выберите нейро-сотрудника"
    )

    name = gr.Textbox(label="Имя нейро-сотрудника", interactive=False)
    prompt = gr.Textbox(label="Промпт нейро-сотрудника", lines=18, interactive=True)
    query = gr.Textbox(label="Запрос пользователя", lines=10, interactive=True)
    link = gr.HTML(label="Ссылка на документ")

    subject.change(
        fn=onchange_model,
        inputs=subject,
        outputs=[name, prompt, query, link]
    )

    demo.load(
        fn=lambda: onchange_model(0),
        inputs=None,
        outputs=[name, prompt, query, link]
    )

    with gr.Row():
        train_btn = gr.Button("Обучить модель", variant="primary")
        request_btn = gr.Button("Запрос к модели", variant="secondary")

    with gr.Row():
        response = gr.Textbox(label="Ответ нейро-сотрудника", lines=22)
        log = gr.Textbox(label="Лог работы RAG-системы", lines=22)

    train_btn.click(
        fn=train_model,
        inputs=subject,
        outputs=log
    )

    request_btn.click(
        fn=ask_model,
        inputs=[prompt, query],
        outputs=[response, log]
    )


demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://caf8e6e80deb5972f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Оптимизация нейро-сотрудника

Я провёл оптимизацию нейро-сотрудника по нескольким направлениям.

### 1. Улучшение промпта

В промпт были добавлены:
- личные данные нейро-сотрудника;
- его роль;
- целевая аудитория;
- обязанности;
- правила поведения;
- строгий формат ответа.

Это нужно, чтобы модель отвечала не как обычный чат-бот, а как конкретный сотрудник с определённой должностью.

### 2. Структурирование базы знаний

Изначально база знаний была обычным плохо структурированным текстом. Затем она была разделена на логические блоки:

- анализ клиентских заявок;
- типы проектов;
- оценка сроков;
- оценка стоимости;
- риски проекта;
- формат ответа клиенту.

Это помогает векторной базе находить более точные фрагменты.

### 3. Двухуровневый формат фрагментов

Каждый фрагмент базы знаний был оформлен в формате:

- заголовок 1 уровня;
- заголовок 2 уровня;
- фрагмент.

Именно в таком виде фрагменты передаются на вход языковой модели.

### 4. Ограничение фантазирования модели

В промпт добавлено правило: если информации недостаточно, модель должна честно написать, что нужно уточнить, а не придумывать точные сроки и стоимость.

Это снижает риск галлюцинаций.

### 5. Единый формат ответа

Для всех ответов задана единая структура:

1. Тип проекта
2. Что хочет клиент
3. Что уже понятно
4. Что нужно уточнить
5. Предлагаемое техническое решение
6. Оценка сложности
7. Примерные сроки
8. Примерная стоимость
9. Возможные риски
10. Готовый ответ клиенту

Благодаря этому ответы стали более стабильными, понятными и пригодными для реального использования во фрилансе.

## Вывод

В результате работы был создан нейро-сотрудник «Нейро-помощник программиста».

Он использует базу знаний из Google Документа, загружает её в векторную базу ChromaDB и отвечает на вопросы пользователя через RAG-подход.

Нейро-сотрудник выполняет свои обязанности:
- анализирует заявки клиентов;
- определяет тип проекта;
- формирует уточняющие вопросы;
- предлагает техническое решение;
- оценивает сроки и стоимость;
- предупреждает о рисках;
- пишет готовый ответ клиенту.

База знаний была сначала создана в плохо структурированном виде, затем структурирована по двухуровневому плану. Также была проведена оптимизация промпта и формата ответа.